# 21 — MedQA accuracy and the retrieval lift

MedQA is a multiple-choice benchmark, so the measure that matches it — and the one
reported in the literature — is accuracy: does the model choose the correct option?
The RAGAS context and relevancy metrics compare the answer against the option text,
which a German guideline corpus cannot reproduce, so they stay low regardless of the
pipeline. This notebook instead has the model pick a letter for each question and
reports two figures:

- **With retrieval** — the model chooses using the passages already retrieved in
  notebook 20 (reused from its results file, so no re-retrieval is needed).
- **Without retrieval** — the model chooses from its own knowledge alone.

The gap between them is the retrieval lift: how much the corpus actually adds over
what the model already knows.

## Setup and data

The retrieved passages come from notebook 20's results file. The four answer options
and the correct letter come from the source benchmark, joined by the open question.

In [ ]:
import os, re, json, time, random, glob
import pandas as pd
from google.colab import drive, userdata
from langchain_openai import ChatOpenAI

drive.mount('/content/drive')
DRIVE_PATH = '/content/drive/MyDrive/'
os.environ["OPENROUTER_API_KEY"] = userdata.get('OPENROUTER_API_KEY')

# The same answering model used for generation, so the only difference between the
# two conditions is the presence of the retrieved passages.
def make_llm(model, max_tokens=8):
    return ChatOpenAI(model=model, api_key=os.environ["OPENROUTER_API_KEY"],
                      base_url="https://openrouter.ai/api/v1", temperature=0,
                      max_tokens=max_tokens, max_retries=6, request_timeout=90)

mistral = make_llm("mistralai/mistral-large")

def safe_invoke(llm, prompt, max_tries=6, base=6):
    for a in range(max_tries):
        try:
            return llm.invoke(prompt).content.strip()
        except Exception as e:
            if a == max_tries-1: raise
            time.sleep(min(base*(2**a)+random.uniform(0,3), 90))

# 1. Retrieved passages from notebook 20 (question text -> its contexts).
RESULTS_FILE = DRIVE_PATH + "AWMF_ONLY_FOCUS_results.json"   # point at DUAL_CORPUS_results.json to score notebook 18
res = json.load(open(RESULTS_FILE))
ctx_map = {q: c for q, c in zip(res["question"], res["contexts"])}
print(f"Loaded {len(ctx_map)} retrieved-context sets from {RESULTS_FILE.split('/')[-1]}")

# 2. The MCQ text (with options) and the correct letter, from the source benchmark.
src_path = glob.glob('/content/drive/MyDrive/**/GP_Top10_Bilingual_Open_EndedQ.csv', recursive=True)[0]
src = pd.read_csv(src_path)
mcq_map  = dict(zip(src['English_Open_Question'], src['English_Question']))    # includes the four options
gold_map = dict(zip(src['English_Open_Question'], src['Correct_Answer']))      # the correct letter
print(f"Loaded options and answer keys from {src_path.split('/')[-1]}")

## Answer the questions and score

Each question is answered twice — once with the retrieved passages, once without —
and the chosen letter is compared to the correct one. Progress is saved so a dropped
session resumes cleanly.

In [ ]:
PROMPT_RAG = """Use the reference passages, together with your own knowledge, to choose the single best answer.
Reply with ONLY the letter: A, B, C, or D.

REFERENCE PASSAGES:
{context}

{question}

Answer (letter only):"""

PROMPT_BASE = """Choose the single best answer to the following multiple-choice question.
Reply with ONLY the letter: A, B, C, or D.

{question}

Answer (letter only):"""

def pick_letter(reply):
    m = re.search(r"[ABCD]", (reply or "").upper())
    return m.group(0) if m else "?"

out_file = DRIVE_PATH + "MEDQA_ACCURACY_results.json"
if os.path.exists(out_file):
    acc = json.load(open(out_file)); done = set(acc["question"])
else:
    acc = {"question":[], "gold":[], "rag":[], "base":[]}; done = set()

questions = [q for q in res["question"] if q in mcq_map and q in gold_map]
print(f"Scoring {len(questions)} questions ({len(done)} already done)")

for i, q in enumerate(questions):
    if q in done: continue
    gold = str(gold_map[q]).strip().upper()
    mcq  = mcq_map[q]
    ctx  = "\n\n".join(ctx_map.get(q, []))
    try:
        rag  = pick_letter(safe_invoke(mistral, PROMPT_RAG.format(context=ctx, question=mcq)))
        base = pick_letter(safe_invoke(mistral, PROMPT_BASE.format(question=mcq)))
        acc["question"].append(q); acc["gold"].append(gold); acc["rag"].append(rag); acc["base"].append(base)
        done.add(q); json.dump(acc, open(out_file, "w"))
        if len(acc["question"]) % 20 == 0: print(f"  {len(acc['question'])}/{len(questions)}")
        time.sleep(1.0)
    except Exception as e:
        print("err", i, str(e)[:90]); time.sleep(6)

# Score.
n = len(acc["question"])
rag_correct  = sum(1 for g, r in zip(acc["gold"], acc["rag"])  if g == r)
base_correct = sum(1 for g, b in zip(acc["gold"], acc["base"]) if g == b)
print("\n=== MedQA accuracy (n =", n, ") ===")
print(f"With retrieval (RAG):   {rag_correct/n:.3f}  ({rag_correct}/{n})")
print(f"Without retrieval:      {base_correct/n:.3f}  ({base_correct}/{n})")
print(f"Retrieval lift:         {(rag_correct-base_correct)/n:+.3f}")

## Reading the result

If the two accuracies are close, the model already knew these answers and the corpus
adds little on this benchmark — expected, since MedQA is board-exam material the model
has largely seen. A positive lift means the passages helped on questions it would
otherwise miss; a negative lift means the passages distracted it. Either way this is
the number that reflects "does the system answer the question correctly", and it is
directly comparable to published MedQA figures.